# 17 — Kinematic phase space

Ligand drift vs COM (centre-of-mass) displacement scatter, per target. Bottom-left = tight binder (never moves). Top-right = escaper (leaves the pocket). This is the visual sanity check that "actives cluster in the stable quadrant" before Act 4 asks whether the clustering is strong enough to rank on.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/17_kinematic_phase_space_figK.png`.)_


> **Reader guide.** *Experiment A3:* 2-D phase-space projection of kinematic features.
>
> **Method:** per-target scatter of drift vs COM displacement; label-coloured (active / non-binder).
>
> **Reproducibility contract:** reads `data/derived/features.parquet` + metadata.

In [ ]:
# --- notebook preamble ---
NB_STEM = "33_kinematic_phase_space"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 5. Ligand kinematic phase space — drift vs escape

Kinematic phase space = drift on one axis, escape on the other. Two features per target:
- **`lig_drift_mean_A`** (x) — how much the ligand rotates/translates *inside* the pocket after alignment.
- **`lig_com_disp_max_A`** (y) — the biggest excursion of the ligand COM from its initial position.

Reading the quadrants:
- **bottom-left** = tight binder (never moves).
- **top-right** = escaper (leaves the pocket).
- **top-left** = ligand rocks in place but stays (fine for weak binders in wide pockets).
- **bottom-right** = ligand translates as a unit into another site. Rare.

Dotted lines at 3 Å split the plane into the four quadrants.


In [ ]:

targets = sorted(df.target.unique())
n = len(targets); nc = 3; nr = (n + nc - 1) // nc
fig, axes = plt.subplots(nr, nc, figsize=(4.5*nc, 3.6*nr))
for ax, tgt in zip(np.array(axes).ravel(), targets):
    sub = df[df.target == tgt]
    if 'is_active' in df:
        for is_act, color, marker, lab in [(False, DECOY_C, 'o', 'decoy'),
                                            (True,  ACTIVE_C, '^', 'active')]:
            s = sub[sub.is_active == is_act]
            ax.scatter(s.lig_drift_mean_A, s.lig_com_disp_max_A,
                       c=color, marker=marker, s=70 if is_act else 45,
                       alpha=0.85 if is_act else 0.6,
                       edgecolors=NAVY, linewidths=0.5, label=lab)
        ax.legend(fontsize=7, loc='best')
    else:
        ax.scatter(sub.lig_drift_mean_A, sub.lig_com_disp_max_A, c=NAVY, s=50, alpha=0.7)
    ax.axvline(3, color=GREYD, ls=':', lw=1); ax.axhline(3, color=GREYD, ls=':', lw=1)
    ax.set_title(tgt); ax.set_xlabel('mean pose drift [Å]'); ax.set_ylabel('max COM disp [Å]')
    ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.4)
for ax in np.array(axes).ravel()[len(targets):]:
    ax.axis('off')
plt.tight_layout()

**What to look for.** Actives should over-populate the bottom-left quadrant (drift < 3 Å AND max-disp < 3 Å). If a target's actives are spread across the plane while decoys sit tight, the docking pose was probably wrong for the actives — bad box geometry or starting-pose bias. If both classes sit tight everywhere, kinematics won't discriminate on that target. Lean on `hb_persistence_frac`, `vdw_contacts`, or IFP features instead.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
